In [0]:
-- in type 1 we add new records and update old records without keeping history
merge into data.ipldata.customers_dim t
using data.ipldata.customers s
on t.customer_id = s.customer_id
when matched then
  update set
    t.email = s.email,
    t.city  = s.city
when not matched then
  insert (customer_id, name, email, city)
  values (s.customer_id, s.name, s.email, s.city);

-- update old records
merge into data.ipldata.customers_contact_dim t
using data.ipldata.customers s
on t.customer_id = s.customer_id and t.is_active = true
when matched and t.phone_number != s.phone_number then
  update set
    t.end_date = current_timestamp(),
    t.is_active = false;

-- insert new row
insert into data.ipldata.customers_contact_dim
   (customer_id, phone_number, start_date, end_date, is_active)
select s.customer_id, s.phone_number, current_timestamp(),NULL,true
from data.ipldata.customers s
left join data.ipldata.customers_contact_dim t
  on s.customer_id = t.customer_id
  and t.is_active = true
where
  t.customer_id is null or t.phone_number != s.phone_number; 